# Kaggle GPU Runner for TDR Transmission Line Fault & Load Identification

**EE 305 Electromagnetic Waves Project — Theme 4**  
*Course: EE 305 Electromagnetic Waves, IIT Indore*  
*Project Repository: ML-Based TDR Fault & Load Identification*

---

### ⚠️ Required Kaggle Notebook Settings
- **Accelerator:** GPU (NVIDIA T4 or P100)
- **Internet:** **On** (required to clone repository and install lightweight packages)
- **Persistence:** Files persist in `/kaggle/working/`

This notebook executes the full experimental pipeline (`--mode full`) on a Kaggle GPU:
1. Clones the repository into `/kaggle/working/repo`
2. Installs lightweight dependencies (`h5py`, `pyyaml`, `lightgbm`, `shap`, `pyarrow`, `tabulate`)
3. Executes physics and pipeline tests (`pytest -q tests.py`)
4. Generates full datasets (85,000 samples) and trains all models (Conventional, LightGBM, ResNet1D multi-task, plus ablations and learning curves)
5. Executes evaluation suite E1–E8 and renders all 16 publication-grade figures
6. Compresses all outputs into `/kaggle/working/outputs.zip` for one-click local download


In [1]:
# 1. Clone Repository (Replace <USERNAME> with your GitHub username)
REPO_URL = "https://github.com/ys-0212/gp15_em_waves"

import os
import shutil
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    print(f"Removing existing {repo_dir} for a clean clone...")
    shutil.rmtree(repo_dir)

!git clone $REPO_URL {repo_dir}
%cd {repo_dir}


Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 297, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 297 (delta 101), reused 297 (delta 101), pack-reused 0 (from 0)
Receiving objects: 100% (297/297), 48.30 MiB | 32.49 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/kaggle/working/repo


In [2]:
# 2. Install Lightweight Dependencies & Check GPU Environment
!pip install -q h5py pyyaml lightgbm shap pyarrow tabulate

import torch
import numpy as np

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:    {torch.cuda.device_count()}")
print("=" * 60)


PyTorch Version: 2.11.0+cu130
CUDA Available:  True
GPU Device:      Tesla T4
Device Count:    2


In [3]:
# 3. Verify Test Suite (15 physics, pipeline and baseline tests)
!pytest -q tests.py


...............                                                          [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/l

In [4]:
# 4. Master Orchestrator: Full Dataset Generation, Training & Evaluation
# Running in full mode on GPU with --skip-existing to allow resume if interrupted
!python run_all.py --mode full --device cuda --skip-existing


MASTER TDR PIPELINE: MODE=FULL, DEVICE=cuda
STAGES: ['gen', 'validate', 'conventional', 'lgbm', 'resnet', 'evaluate', 'figures']

##### STAGE: gen  (pipeline elapsed 0s)

>>> Running: /usr/bin/python3 2_generate_data.py
Generating datasets (mode: FULL)...
Generating train: N=60000, seed=1...
  Done in 116.50s (1.94s per 1k)
Generating val: N=10000, seed=2...
  Done in 19.43s (1.94s per 1k)
Generating test: N=10000, seed=3...
  Done in 19.28s (1.93s per 1k)
Generating OOD subsets: 5 subsets x 1000 samples...
  OOD Done in 8.92s (1.78s per 1k)

All datasets generated in 164.95s.
Dataset card written to data/dataset_card.md
Debug verification plot saved to results/debug/debug_fault_gallery.png
>>> Finished in 171.47s | pipeline elapsed 2.9 min

##### STAGE: validate  (pipeline elapsed 171s)
Skipping 'validate' (sim_validation.json already exists)

##### STAGE: conventional  (pipeline elapsed 171s)

>>> Running: /usr/bin/python3 3_train.py --model conventional

--- Running Conventional TDR

In [5]:
# 5. Package Outputs into outputs.zip and Print Main Comparison
import os
import zipfile
from pathlib import Path

zip_dest = Path("/kaggle/working/outputs.zip")
dirs_to_zip = [
    "results",
    "figures",
    "app/weights",
    "report/figures",
    "report/tables",
]
files_to_zip = [
    "data/dataset_card.md",
]

print(f"Packaging artifacts into {zip_dest}...")
with zipfile.ZipFile(zip_dest, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in dirs_to_zip:
        p = Path(d)
        if p.exists():
            for f in p.rglob("*"):
                if f.is_file() and not f.name.endswith(".pt_tmp"):
                    rel = f.relative_to(Path("."))
                    zf.write(f, rel)
    for f_str in files_to_zip:
        f = Path(f_str)
        if f.exists():
            zf.write(f, f.relative_to(Path(".")))

zip_size_mb = zip_dest.stat().st_size / (1024 * 1024)
print(f"Outputs successfully zipped! Size: {zip_size_mb:.2f} MB")
print(f"Download path: {zip_dest}")

# Display key comparison table
comparison_md = Path("results/tables/main_comparison.md")
if comparison_md.exists():
    print("\n" + "=" * 60)
    print("FINAL EXPERIMENTAL RESULTS (results/tables/main_comparison.md)")
    print("=" * 60)
    print(comparison_md.read_text(encoding="utf-8"))


Packaging artifacts into /kaggle/working/outputs.zip...
Outputs successfully zipped! Size: 339.05 MB
Download path: /kaggle/working/outputs.zip

FINAL EXPERIMENTAL RESULTS (results/tables/main_comparison.md)
| Model               | Fault Acc (%)   | Fault Macro-F1   | Loc MAE (m)   | Loc Median (m)   | Loc P90 (m)   | Loc <1m (%)   | Load Acc (%)   | Load Macro-F1   | MAPE R (%)   | MAPE L (%)   | MAPE C (%)   |
|:--------------------|:----------------|:-----------------|:--------------|:-----------------|:--------------|:--------------|:---------------|:----------------|:-------------|:-------------|:-------------|
| Conventional TDR    | 61.83           | 0.6224           | 9.47          | 0.91             | 36.06         | 51.7          | 80.19          | 0.8123          | 38.2         | 134.3        | 140.5        |
| LightGBM            | 78.27 ± 0.07    | 0.7902 ± 0.0009  | 7.67 ± 0.04   | 1.42 ± 0.02      | 25.66 ± 0.29  | 42.5 ± 0.2    | 93.40 ± 0.11   | 0.9348 ± 0.0011 | 12.1 